# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iankitthakur/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip -q install duckdb
import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

frame = con.sql("""
SELECT
  client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS early_impressions,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS early_clicks,
  SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15') * 1.0
    / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15'), 0) AS early_ctr,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS early_avg_position,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS early_active_days,
  SUM(gsc_impressions)  FILTER (WHERE report_date >  DATE '2026-03-15') AS late_impressions
FROM march
WHERE gsc_data_available IS TRUE
  AND hash(content_hash_id) % 50 = 0
GROUP BY 1, 2
""").df()

df = frame[frame["early_impressions"] >= 20].copy()
df["late_impressions"] = df["late_impressions"].fillna(0)
# label: used ONLY to test signals, never as an input
df["label_declining"] = ((df["late_impressions"] / 16) < 0.8 * (df["early_impressions"] / 15)).astype(int)
df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
y = df["label_declining"]
print(df.shape, "declining rate:", round(y.mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2191, 9) declining rate: 0.329


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
cols = ["early_impressions", "early_clicks", "early_ctr", "early_avg_position", "early_active_days"]
display(df[cols].describe(percentiles=[.5, .9, .99]).round(3).T)
print("pages with zero early clicks:", int((df["early_clicks"] == 0).sum()), f"({round((df['early_clicks'] == 0).mean() * 100, 1)}%)")

,count,mean,std,min,50%,90%,99%,max
early_impressions,2191.0,1209.389,3331.591,20.000,278.00,3024.000,12607.200,86860.000
early_clicks,2191.0,3.424,11.405,0.000,0.00,8.000,48.100,216.000
early_ctr,2191.0,0.003,0.006,0.000,0.00,0.008,0.028,0.074
early_avg_position,2191.0,14.924,15.511,0.336,8.58,35.150,74.414,102.888
early_active_days,2191.0,13.719,2.420,2.000,15.00,15.000,15.000,15.000


pages with zero early clicks: 1198 (54.7%)


Distributions (observed): the fields have heavy tails. Early impressions average about 1,209 per page but reach 86,860; early clicks average about 3.4 and reach 216; CTR averages about 0.3% and reaches 7.4%; average position averages about 15 and reaches about 103. A few large pages dominate totals, so I use medians and group counts (with n) instead of averages across all pages. Many pages also have zero clicks in the first half of March (the share is printed above).

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
# Signal 1: position vs CTR
d = df.dropna(subset=["early_avg_position", "early_ctr"]).copy()
d["pos_bucket"] = pd.cut(d["early_avg_position"], [0, 3, 10, 20, 100], labels=["1-3", "4-10", "11-20", "21+"])
g = d.groupby("pos_bucket", observed=True)
print("SIGNAL 1: position vs CTR")
display(pd.DataFrame({
    "n": g["content_hash_id"].count(),
    "median_ctr_pct": (g["early_ctr"].median() * 100).round(3),
    "pooled_ctr_pct": (g["early_clicks"].sum() / g["early_impressions"].sum() * 100).round(3),
    "declining_rate": g["label_declining"].mean().round(3)}))

# Signal 2: volume vs decline
d2 = df.copy()
d2["vol_bucket"] = pd.qcut(d2["early_impressions"], 4, duplicates="drop")
print("SIGNAL 2: volume vs decline")
display(d2.groupby("vol_bucket", observed=True).agg(n=("label_declining", "size"), declining_rate=("label_declining", "mean")).round(3))

# Signal 3: active days vs decline
d3 = df.copy()
d3["active_bucket"] = pd.cut(d3["early_active_days"], [-1, 7, 14, 15], labels=["0-7 days", "8-14 days", "15 days"])
print("SIGNAL 3: active days vs decline")
display(d3.groupby("active_bucket", observed=True).agg(n=("label_declining", "size"), declining_rate=("label_declining", "mean")).round(3))
print("base rate:", round(y.mean(), 3))

SIGNAL 1: position vs CTR


,n,median_ctr_pct,pooled_ctr_pct,declining_rate
pos_bucket,,,,
1-3,219,0.225,0.466,0.311
4-10,993,0.065,0.291,0.354
11-20,440,0.000,0.405,0.323
21+,538,0.000,0.120,0.292


SIGNAL 2: volume vs decline


,n,declining_rate
vol_bucket,,
"(19.999, 83.0]",553,0.335
"(83.0, 278.0]",544,0.316
"(278.0, 997.0]",546,0.310
"(997.0, 86860.0]",548,0.354


SIGNAL 3: active days vs decline


,n,declining_rate
active_bucket,,
0-7 days,87,0.149
8-14 days,696,0.297
15 days,1408,0.355


base rate: 0.329


**Signal 1: position vs CTR.** Claim: pages ranking closer to the top get a higher CTR. **Verdict: MIXED.** Pooled CTR is highest at positions 1–3 (about 0.47%) and lowest at 21+ (about 0.12%), and median CTR also falls, but the 11–20 group (about 0.41%) is above the 4–10 group (about 0.29%), so the pattern is not steady. The declining rate by position (about 0.29 to 0.35) shows no pattern.

**Signal 2: volume vs decline.** Claim: pages with more impressions are less likely to decline. **Verdict: MIXED.** The declining rate across the four volume groups is about 0.31 to 0.35 with about 550 pages each, with no steady direction, and the highest-volume group is the highest.

**Signal 3: active days vs decline.** Claim: pages active on fewer days in the first half of March are more at risk. **Verdict: OPPOSITE.** Pages with 0–7 active days declined at about 15% (n = 87) against about 36% for pages active on all 15 days. One possibility is that pages active on few days were new or just starting to get impressions, so their traffic grew; I did not check this.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [8]:
r = df.dropna(subset=["early_avg_position", "early_ctr"]).copy()
r["pos_bucket"] = pd.cut(r["early_avg_position"], [0, 3, 10, 20, 1000], labels=["1-3", "4-10", "11-20", "21+"]).astype(str)
sums = r.groupby("pos_bucket")[["early_clicks", "early_impressions"]].sum()
r["peer_ctr"] = r["pos_bucket"].map(sums["early_clicks"] / sums["early_impressions"])
r["clicks_below_peers"] = (r["peer_ctr"] * r["early_impressions"] - r["early_clicks"]).clip(lower=0)
r = r[r["early_impressions"] >= 100]
r["group"] = np.where(r["clicks_below_peers"] > 0, "CTR below peers at same position", "CTR at or above peers")
display(r.groupby("group").agg(n=("label_declining", "size"), declining_rate=("label_declining", "mean")).round(3))
print("base rate:", round(y.mean(), 3))

,n,declining_rate
group,,
CTR at or above peers,514,0.243
CTR below peers at same position,1042,0.367


base rate: 0.329


Flag-linked test (the CTR-fix logic, observed): the flag assumes that a page whose CTR is below its peers at the same position is a candidate for a title or meta rewrite. In my slice (pages with at least 100 early impressions), pages with CTR below peers declined more often (about 37%) than pages at or above peer CTR (about 24%), a gap of about 13 points, against a 32.9% base rate. That supports using the flag to decide which pages to look at first. It does not show that the CTR gap causes the decline, or that rewriting a title would fix it.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


For a content team (decision-support): none of the simple signals I tested (position, volume, active days) predicts decline well on its own, so do not triage on one of them. The one pattern that held up is CTR below peers at the same position, and pages like that are a sensible first batch to open and look at. Treat every flag as a reason to look, not a conclusion, and have a person decide what to change.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.